# Annotated Mozart Piano Sonatas (DCML v2.3) to TiLiA

Full run: download the pinned files (checked by md5), convert the 54 movements, run TiLiA, build the package. Output directory: `$DATASET_CONVERTERS_OUT`. Needs `TILIA` pointing at TiLiA's command line and format-converters installed.

In [ ]:
import importlib.util, os, sys
from pathlib import Path

root = Path.cwd()
while not (root / "common").is_dir():
    root = root.parent
sys.path.insert(0, str(root))
spec = importlib.util.spec_from_file_location("dcml_convert", root / "dcml-mozart" / "convert.py")
convert = importlib.util.module_from_spec(spec)
spec.loader.exec_module(convert)

out = Path(os.environ["DATASET_CONVERTERS_OUT"])
out.mkdir(parents=True, exist_ok=True)

## Download

There is no single archive: each of the 109 files (metadata.tsv, and a harmonies and a measures table per movement) has its md5 pinned in `md5s.json`, and `fetch` rejects a mismatch.

In [ ]:
files = convert.fetch_sources()
print(len(files), "files, all checked")

## Convert and run TiLiA

In [ ]:
summary = convert.convert(convert.PIECES, out, files=files)

## Package

In [ ]:
print(convert.package(out, out / "package"))

## Summary

Components in the saved files against rows counted from the source tables.

In [ ]:
totals = {}
for layers in summary.values():
    for layer, v in layers.items():
        t = totals.setdefault(layer, [0, 0])
        t[0] += v["components"]; t[1] += v["source_rows"]
print(len(summary), "movements")
for layer, (comps, rows) in totals.items():
    print(f"{layer:16} components {comps:6}  source rows {rows:6}")
assert all(v["components"] == v["source_rows"] for l in summary.values() for v in l.values())